# A03 平台主键证据复核

只读取同目录已捕获的 Smartbi SQL 结果、SQL 和 DOM，不直接连接数据库。平台原查询见 A03_PK_ALL_TABLES_20260904.sql。执行时间见结果 JSON，所有表采用原始主键分组，不做关联、不抽样、不先去重。


In [ ]:
from pathlib import Path
import json, hashlib
p = Path('.')
data = json.loads((p/'A03_PK_PLATFORM_RESULTS_20260904.json').read_text(encoding='utf-8'))
targets = json.loads((p/'A03_PK_TARGETS_20260904.json').read_text(encoding='utf-8'))
dom = (p/'A03_PK_PLATFORM_DOM_20260904.txt').read_text(encoding='utf-8')
assert hashlib.sha256((p/'A03_PK_ALL_TABLES_20260904.sql').read_bytes()).hexdigest() == data['sqlSha256']
assert len(data['rows']) == data['platformReportedTotal'] == 21
assert len({r[1] for r in data['rows']}) == 21
for target in targets:
    row = next(r for r in data['rows'] if r[1] == target['object'])
    assert '- row "' + ' '.join(row) + '"' in dom
    nums = [int(str(v).replace(',', '')) for v in row[2:]]
    assert nums[0] == nums[1] == target['expectedRows']
    assert nums[2:] == [0,0,0,0]
assert sum(int(r[2].replace(',','')) for r in data['rows'] if int(r[0]) <= 18) == 313593
print('21/21 平台主键聚合证据一致；类型、资源锁、B独立复核未由此签收。')
